In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [30]:
backinh_path = 'BacKinh.csv'
hcm_path = 'hcm_aqi_full_dataset.csv'
folder_path = 'dataset'

file_path_backinh = os.path.join(folder_path, backinh_path)
file_path_hcm = os.path.join(folder_path, hcm_path)

df_backinh = pd.read_csv(file_path_backinh) 
df_hcm = pd.read_csv(file_path_hcm)

df_hcm.head(10)


,time,PM10,PM2.5,CO,NO2,O3,SO2,AQI,UV,Temperature,Humidity,Rain,Wind_Speed,Wind_Dir
0,2023-01-01 00:00:00,95.0,65.0,993.0,84.4,25.0,37.5,133,0.00,23.5,65,0.0,11.3,9
1,2023-01-01 01:00:00,86.1,59.0,846.0,70.2,28.0,30.8,133,0.00,23.0,67,0.0,10.3,12
2,2023-01-01 02:00:00,83.4,57.0,821.0,65.8,26.0,28.9,132,0.00,22.5,70,0.0,7.9,360
3,2023-01-01 03:00:00,80.1,54.8,834.0,64.0,22.0,28.5,131,0.00,22.0,73,0.0,8.3,360
4,2023-01-01 04:00:00,69.4,47.5,838.0,60.1,20.0,27.2,129,0.00,21.9,72,0.0,7.9,357
5,2023-01-01 05:00:00,57.9,39.5,805.0,53.7,22.0,24.5,128,0.00,22.4,69,0.0,7.2,354
6,2023-01-01 06:00:00,48.8,33.2,759.0,46.0,26.0,21.0,126,0.00,22.2,71,0.0,8.7,7
7,2023-01-01 07:00:00,31.5,21.5,555.0,23.3,48.0,6.7,124,0.15,23.0,80,0.0,6.7,16
8,2023-01-01 08:00:00,29.9,20.3,529.0,19.3,56.0,6.4,124,0.85,23.3,78,0.3,6.8,342
9,2023-01-01 09:00:00,29.2,19.6,492.0,13.6,66.0,6.2,124,1.65,24.3,75,0.0,4.4,351


In [39]:
# --- CELL: TÁI HIỆN (REPRODUCE) V2 - TIỆM CẬN 0.912 ---
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler # Đổi sang MinMaxScaler theo paper
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import time

print("🎯 Đang tinh chỉnh để tiệm cận kết quả bài báo...")

# 1. Đọc data (giữ nguyên bước dọn dẹp cũ)
df_bk = df_backinh.copy()
df_bk['pm2.5'] = df_bk['pm2.5'].interpolate(method='linear')
df_bk = df_bk.dropna(subset=['pm2.5']).reset_index(drop=True)
df_bk['time'] = pd.to_datetime(df_bk[['year', 'month', 'day', 'hour']])
df_bk = pd.get_dummies(df_bk, columns=['cbwd'], prefix='wind_dir')

# 2. Tạo 24 Lags (Window size = 24 chuẩn paper)
n_lags = 24
lag_cols = ['pm2.5', 'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir']
new_features = {}
for col in lag_cols:
    for i in range(1, n_lags + 1):
        new_features[f'{col}_lag_{i}'] = df_bk[col].shift(i)
new_features['pm2.5_t+1'] = df_bk['pm2.5'].shift(-1)

df_repro = pd.concat([df_bk, pd.DataFrame(new_features)], axis=1).dropna().reset_index(drop=True)

# 3. Chia Train/Test (2010-2013 / 2014)
train_df = df_repro[df_repro['time'].dt.year < 2014]
test_df = df_repro[df_repro['time'].dt.year == 2014]

drop_cols = ['time', 'pm2.5', 'pm2.5_t+1', 'year', 'month', 'day', 'hour']
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_train = train_df['pm2.5_t+1'].values
y_test = test_df['pm2.5_t+1'].values

# 4. Chuẩn hóa về [0, 1] theo đúng văn bản của bài báo
preprocessor = ColumnTransformer([
    ('scale', MinMaxScaler(), X_train.columns) # Scale tất cả về [0, 1]
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# 5. Huấn luyện XGBoost với tham số "ép xung" nhẹ
# Vẫn giữ 200 cây nhưng tăng độ sâu để xử lý lượng feature lớn
xgb_repro = XGBRegressor(
    n_estimators=200,      # Chuẩn paper
    max_depth=8,           # Tăng độ sâu để học kỹ hơn
    learning_rate=0.1,     # Tăng nhẹ tốc độ học
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    tree_method='hist',
    device='cuda'
)

xgb_repro.fit(X_train_scaled, y_train)

# 6. Kiểm tra kết quả
y_pred = xgb_repro.predict(X_test_scaled)
r2_repro = r2_score(y_test, y_pred)

print("-" * 50)
print(f"Kết quả:")
print(f"Điểm R2 bài báo: 0.9120")
print(f"Điểm R2 hiện tại: {r2_repro:.4f}")
print("-" * 50)

🎯 Đang tinh chỉnh để tiệm cận kết quả bài báo...


c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:57:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:57:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


--------------------------------------------------
Kết quả:
Điểm R2 bài báo: 0.9120
Điểm R2 hiện tại: 0.8756
--------------------------------------------------


In [41]:
# --- CELL: CHẠY CẤU HÌNH BÀI BÁO (BASELINE 24-LAG) CHO DATA HCM ---
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import time

print("🚀 Đang áp dụng cấu hình bài báo quốc tế (24 Lags) cho dữ liệu HCM...")

# 1. Đọc và dọn dẹp data HCM
df_hcm_copy = df_hcm.copy()
pollutants_hcm = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2']

for col in pollutants_hcm:
    if col in df_hcm_copy.columns:
        df_hcm_copy[col] = df_hcm_copy[col].interpolate(method='linear')

df_hcm_copy = df_hcm_copy.dropna(subset=['PM2.5']).reset_index(drop=True)

# 2. Tạo 24 Lags (Window size = 24 chuẩn paper) [cite: 303]
n_lags = 24
new_features_hcm = {}
# Tạo lags cho toàn bộ các chất ô nhiễm hiện có
for col in pollutants_hcm:
    if col in df_hcm_copy.columns:
        for i in range(1, n_lags + 1):
            new_features_hcm[f'{col}_lag_{i}'] = df_hcm_copy[col].shift(i)

# Target: 1 giờ tiếp theo (t+1)
new_features_hcm['PM2.5_t+1'] = df_hcm_copy['PM2.5'].shift(-1)

df_hcm_repro = pd.concat([df_hcm_copy, pd.DataFrame(new_features_hcm)], axis=1).dropna().reset_index(drop=True)

# 3. Chia Train/Test (80% / 20% vì HCM không chia theo năm như Bắc Kinh)
train_size = int(len(df_hcm_repro) * 0.8)
train_hcm = df_hcm_repro.iloc[:train_size]
test_hcm = df_hcm_repro.iloc[train_size:]

# Loại bỏ các cột không phải đặc trưng huấn luyện
drop_cols = ['time', 'PM2.5', 'PM2.5_t+1'] + pollutants_hcm
X_train_hcm = train_hcm.drop(columns=[c for c in drop_cols if c in train_hcm.columns])
X_test_hcm = test_hcm.drop(columns=[c for c in drop_cols if c in test_hcm.columns])
y_train_hcm = train_hcm['PM2.5_t+1'].values
y_test_hcm = test_hcm['PM2.5_t+1'].values

# 4. Chuẩn hóa về [0, 1] y hệt bài báo [cite: 302]
preprocessor_hcm = ColumnTransformer([
    ('scale', MinMaxScaler(), X_train_hcm.columns)
])

X_train_scaled = preprocessor_hcm.fit_transform(X_train_hcm)
X_test_scaled = preprocessor_hcm.transform(X_test_hcm)

# 5. Huấn luyện XGBoost với cấu hình "Baseline quốc tế" [cite: 309]
xgb_hcm_paper = XGBRegressor(
    n_estimators=200,      # Chuẩn paper
    max_depth=8,           # Độ sâu đã tinh chỉnh
    learning_rate=0.1, 
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    tree_method='hist',
    device='cuda'
)

start_time = time.time()
xgb_hcm_paper.fit(X_train_scaled, y_train_hcm)
print(f"⏱️ Huấn luyện xong trong: {time.time() - start_time:.2f} giây")

# 6. Kiểm tra kết quả
y_pred_hcm = xgb_hcm_paper.predict(X_test_scaled)
r2_hcm = r2_score(y_test_hcm, y_pred_hcm)

print("-" * 50)
print(f"KẾT QUẢ ÁP DỤNG CÔNG THỨC QUỐC TẾ LÊN DATA HCM:")
print(f"Điểm R2 đạt được: {r2_hcm:.4f}")
print(f"So với điểm R2 Bắc Kinh trong paper: 0.9120")
print("-" * 50)

🚀 Đang áp dụng cấu hình bài báo quốc tế (24 Lags) cho dữ liệu HCM...


c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [12:01:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [12:01:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


⏱️ Huấn luyện xong trong: 6.88 giây
--------------------------------------------------
KẾT QUẢ ÁP DỤNG CÔNG THỨC QUỐC TẾ LÊN DATA HCM:
Điểm R2 đạt được: 0.8382
So với điểm R2 Bắc Kinh trong paper: 0.9120
--------------------------------------------------


In [12]:
# --- CELL 1 (BASELINE): TIỀN XỬ LÝ DỮ LIỆU BẮC KINH CƠ BẢN ---
import pandas as pd
import numpy as np

print("⏳ Đang tiền xử lý df_backinh (Phiên bản Cơ bản - Baseline)...")

# 1. Gom ngày tháng năm thành 1 cột datetime (nếu có)
if 'year' in df_backinh.columns:
    df_backinh['time'] = pd.to_datetime(df_backinh[['year', 'month', 'day', 'hour']])
    df_backinh = df_backinh.drop(columns=['No', 'year', 'month', 'day', 'hour'], errors='ignore')

# 2. Xử lý NaN Cơ bản (Nội suy lấp đầy PM2.5 và bỏ các dòng rác đầu tiên)
df_backinh['pm2.5'] = df_backinh['pm2.5'].interpolate(method='linear')
df_backinh = df_backinh.dropna(subset=['pm2.5']).reset_index(drop=True)

# 3. Mã hóa One-Hot cho Hướng gió (cbwd) để model hiểu được chữ
if 'cbwd' in df_backinh.columns:
    df_backinh = pd.get_dummies(df_backinh, columns=['cbwd'], prefix='wind_dir')

# 4. CHỈ TẠO TARGET ĐỂ DỰ ĐOÁN (Không tạo 24 Lags)
n_steps = 3  
new_targets = {}
for i in range(1, n_steps + 1):
    new_targets[f'pm2.5_t+{i}'] = df_backinh['pm2.5'].shift(-i)

# Ghép target vào data và xóa 3 dòng cuối cùng bị rỗng (do shift)
df_bk_baseline = pd.concat([df_backinh, pd.DataFrame(new_targets)], axis=1)
df_bk_baseline = df_bk_baseline.dropna().reset_index(drop=True)

# 5. CHIA TRAIN/TEST (Train: 2010-2013, Test: 2014 chuẩn bài báo)
train_bk = df_bk_baseline[df_bk_baseline['time'].dt.year < 2014]
test_bk = df_bk_baseline[df_bk_baseline['time'].dt.year == 2014]

target_cols_bk = ['pm2.5_t+1', 'pm2.5_t+2', 'pm2.5_t+3']
# Bỏ cột time (vì model không đọc được), Cột 'pm2.5' ở giờ hiện tại VẪN ĐƯỢC GIỮ LẠI làm feature
drop_cols_bk = target_cols_bk + ['time']

X_train_bk_base = train_bk.drop(columns=drop_cols_bk)
X_test_bk_base = test_bk.drop(columns=drop_cols_bk)

y_train_bk_base = train_bk[target_cols_bk].values
y_test_bk_base = test_bk[target_cols_bk].values

print(f"✅ BẮC KINH (BASELINE) XONG!")
print(f"📈 Tập Train: {X_train_bk_base.shape}")
print(f"📉 Tập Test:  {X_test_bk_base.shape}")

⏳ Đang tiền xử lý df_backinh (Phiên bản Cơ bản - Baseline)...
✅ BẮC KINH (BASELINE) XONG!
📈 Tập Train: (35040, 11)
📉 Tập Test:  (8757, 11)


In [13]:
# --- CELL 2 (BASELINE): TIỀN XỬ LÝ DỮ LIỆU HỒ CHÍ MINH CƠ BẢN ---
import pandas as pd
import numpy as np

print("⏳ Đang tiền xử lý df_hcm (Phiên bản Cơ bản - Baseline)...")

# 1. Dọn dẹp NaN (Lấp đầy bằng nội suy)
# Đảm bảo các cột chất ô nhiễm không bị thủng lỗ
pollutants_hcm = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2']
for col in pollutants_hcm:
    if col in df_hcm.columns:
        df_hcm[col] = df_hcm[col].interpolate(method='linear')

df_hcm = df_hcm.dropna(subset=['PM2.5']).reset_index(drop=True)

# 2. CHỈ TẠO TARGET ĐỂ DỰ ĐOÁN (Không tạo 22 Lags đa biến)
n_steps_hcm = 3  
new_targets_hcm = {}

for i in range(1, n_steps_hcm + 1):
    new_targets_hcm[f'PM2.5_t+{i}'] = df_hcm['PM2.5'].shift(-i)

# Ghép target vào data gốc và xóa 3 dòng cuối cùng bị rỗng (do lệnh shift)
df_hcm_baseline = pd.concat([df_hcm, pd.DataFrame(new_targets_hcm)], axis=1)
df_hcm_baseline = df_hcm_baseline.dropna().reset_index(drop=True)

# 3. CHIA TRAIN / TEST (80% / 20%)
train_size = int(len(df_hcm_baseline) * 0.8)
train_hcm = df_hcm_baseline.iloc[:train_size]
test_hcm = df_hcm_baseline.iloc[train_size:]

target_cols_hcm = ['PM2.5_t+1', 'PM2.5_t+2', 'PM2.5_t+3']
# Bỏ cột time (vì model không đọc được), Cột 'PM2.5' ở giờ hiện tại VẪN ĐƯỢC GIỮ LẠI làm feature
drop_cols_hcm = target_cols_hcm + ['time']

X_train_hcm_base = train_hcm.drop(columns=[c for c in drop_cols_hcm if c in train_hcm.columns])
X_test_hcm_base = test_hcm.drop(columns=[c for c in drop_cols_hcm if c in test_hcm.columns])

y_train_hcm_base = train_hcm[target_cols_hcm].values
y_test_hcm_base = test_hcm[target_cols_hcm].values

print(f"✅ HỒ CHÍ MINH (BASELINE) XONG!")
print(f"📈 Tập Train: {X_train_hcm_base.shape}")
print(f"📉 Tập Test:  {X_test_hcm_base.shape}")

⏳ Đang tiền xử lý df_hcm (Phiên bản Cơ bản - Baseline)...
✅ HỒ CHÍ MINH (BASELINE) XONG!
📈 Tập Train: (21501, 13)
📉 Tập Test:  (5376, 13)


In [14]:
# --- CELL 3 (BASELINE): CHUẨN HÓA DỮ LIỆU BẮC KINH ---
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import joblib
import os

print("⏳ Đang chuẩn hóa dữ liệu Bắc Kinh (Baseline)...")

# 1. Tách các cột One-Hot (hướng gió) để giữ nguyên (passthrough)
cols_passthrough_bk = [c for c in X_train_bk_base.columns if c.startswith('wind_dir')]

# 2. Các cột còn lại (Nhiệt độ, Áp suất, PM2.5 hiện tại...) sẽ bị Scale
cols_to_scale_bk = [c for c in X_train_bk_base.columns if c not in cols_passthrough_bk]

# 3. Lắp ráp bộ máy Preprocessor
preprocessor_bk_base = ColumnTransformer(
    transformers=[
        ('pass', 'passthrough', cols_passthrough_bk),
        ('std', StandardScaler(), cols_to_scale_bk)
    ],
    verbose_feature_names_out=False
)
preprocessor_bk_base.set_output(transform="pandas")

# 4. Transform dữ liệu
X_train_bk_scaled = preprocessor_bk_base.fit_transform(X_train_bk_base)
X_test_bk_scaled = preprocessor_bk_base.transform(X_test_bk_base)

# Lưu bộ Scaler
os.makedirs('models', exist_ok=True)
joblib.dump(preprocessor_bk_base, 'models/preprocessor_bk_base.pkl')

print(f"✅ Xong! Kích thước X_train Bắc Kinh đã scale: {X_train_bk_scaled.shape}")

⏳ Đang chuẩn hóa dữ liệu Bắc Kinh (Baseline)...
✅ Xong! Kích thước X_train Bắc Kinh đã scale: (35040, 11)


In [15]:
# --- CELL 4 (BASELINE): CHUẨN HÓA DỮ LIỆU HỒ CHÍ MINH ---
print("⏳ Đang chuẩn hóa dữ liệu Hồ Chí Minh (Baseline)...")

# Với Baseline HCM hiện tại, tất cả các cột đều là số liên tục (PM2.5, PM10, NO2, CO, SO2)
# Không có cột One-Hot hay Sin/Cos nào, nên scale toàn bộ!
cols_to_scale_hcm = list(X_train_hcm_base.columns)

preprocessor_hcm_base = ColumnTransformer(
    transformers=[
        ('std', StandardScaler(), cols_to_scale_hcm)
    ],
    verbose_feature_names_out=False
)
preprocessor_hcm_base.set_output(transform="pandas")

# Transform dữ liệu
X_train_hcm_scaled = preprocessor_hcm_base.fit_transform(X_train_hcm_base)
X_test_hcm_scaled = preprocessor_hcm_base.transform(X_test_hcm_base)

joblib.dump(preprocessor_hcm_base, 'models/preprocessor_hcm_base.pkl')

print(f"✅ Xong! Kích thước X_train HCM đã scale: {X_train_hcm_scaled.shape}")

⏳ Đang chuẩn hóa dữ liệu Hồ Chí Minh (Baseline)...
✅ Xong! Kích thước X_train HCM đã scale: (21501, 13)


In [16]:
# --- CELL 5 (BASELINE): HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH CƠ SỞ ---
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import time
from IPython.display import display

print("🚀 BẮT ĐẦU TRẬN CHIẾN KHỞI ĐỘNG: MÔ HÌNH XGBOOST MẶC ĐỊNH (BASELINE)...")

# 1. Khởi tạo XGBoost Mặc định 
# Không dùng bộ tham số ép xung, chỉ bật GPU cho nhanh
xgb_baseline = XGBRegressor(
    random_state=42,
    tree_method='hist',
    device='cuda'  # Nếu máy bạn không có GPU NVIDIA thì xóa dòng này
)

model_xgb_bk_base = MultiOutputRegressor(xgb_baseline)
model_xgb_hcm_base = MultiOutputRegressor(xgb_baseline)

# 2. Huấn luyện Baseline cho Bắc Kinh
print("\n🥊 Đang huấn luyện Baseline Bắc Kinh (Dữ liệu không Lags)...")
start_bk = time.time()
model_xgb_bk_base.fit(X_train_bk_scaled, y_train_bk_base)
print(f"⏱️ Bắc Kinh Baseline xong trong: {time.time() - start_bk:.2f} giây!")

# 3. Huấn luyện Baseline cho Hồ Chí Minh
print("🥊 Đang huấn luyện Baseline Hồ Chí Minh (Dữ liệu không Lags)...")
start_hcm = time.time()
model_xgb_hcm_base.fit(X_train_hcm_scaled, y_train_hcm_base)
print(f"⏱️ Hồ Chí Minh Baseline xong trong: {time.time() - start_hcm:.2f} giây!")

# 4. Hàm đánh giá điểm số
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    results = []
    for i in range(3): # t+1, t+2, t+3
        r2 = r2_score(y_test[:, i], y_pred[:, i])
        mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))
        results.append([r2, mae, rmse])
    return results

# Lấy kết quả
res_bk_base = evaluate_model(model_xgb_bk_base, X_test_bk_scaled, y_test_bk_base)
res_hcm_base = evaluate_model(model_xgb_hcm_base, X_test_hcm_scaled, y_test_hcm_base)

# 5. In bảng điểm sàn (Baseline Score)
comparison_base_df = pd.DataFrame({
    'Tầm nhìn': ['t + 1', 't + 2', 't + 3'],
    'R2_BắcKinh (Baseline)': [res_bk_base[0][0], res_bk_base[1][0], res_bk_base[2][0]],
    'R2_HồChíMinh (Baseline)': [res_hcm_base[0][0], res_hcm_base[1][0], res_hcm_base[2][0]],
    'MAE_BắcKinh': [res_bk_base[0][1], res_bk_base[1][1], res_bk_base[2][1]],
    'MAE_HồChíMinh': [res_hcm_base[0][1], res_hcm_base[1][1], res_hcm_base[2][1]],
}).set_index('Tầm nhìn')

print("\n🏆 BẢNG ĐIỂM SÀN (BASELINE SCORE):")
display(comparison_base_df.round(4))

🚀 BẮT ĐẦU TRẬN CHIẾN KHỞI ĐỘNG: MÔ HÌNH XGBOOST MẶC ĐỊNH (BASELINE)...

🥊 Đang huấn luyện Baseline Bắc Kinh (Dữ liệu không Lags)...


c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Dev

⏱️ Bắc Kinh Baseline xong trong: 0.50 giây!
🥊 Đang huấn luyện Baseline Hồ Chí Minh (Dữ liệu không Lags)...


c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\nguye\miniconda3\envs\DAP\lib\site-packages\xgboost\training.py:199: UserWarning: [11:36:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Dev

⏱️ Hồ Chí Minh Baseline xong trong: 0.75 giây!

🏆 BẢNG ĐIỂM SÀN (BASELINE SCORE):


,R2_BắcKinh (Baseline),R2_HồChíMinh (Baseline),MAE_BắcKinh,MAE_HồChíMinh
Tầm nhìn,,,,
t + 1,0.9433,0.9112,12.3691,3.1076
t + 2,0.8687,0.7900,19.6414,5.1032
t + 3,0.7955,0.6709,25.5455,6.5234
